In [ ]:
import copy,random,re,time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sacrebleu
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence,pack_padded_sequence,pad_packed_sequence
from torch.utils.data import DataLoader

In [ ]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
P_TR="../en_fr_train.csv"
P_VA="../en_fr_val.csv"
P_TE="../en_fr_test.csv"
N=200000
MAXLEN=40
DMAX=100
MINF=3
BS=128
EPOCHS=15
PAT=3
LR=1e-3
D=128
H=256
L=2
DR=0.3
PAD,SOS,EOS,UNK=0,1,2,3
DEV="cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
tok=lambda s:" ".join(re.findall(r"\w+|[^\w\s]",str(s).lower()))
def ld(p):
    d=pd.read_csv(p)[["en","fr"]].dropna()
    d["en"]=d["en"].map(tok)
    d["fr"]=d["fr"].map(tok)
    return d[(d.en!="")&(d.fr!="")]
def cap(d):
    a=d.en.str.split().str.len()
    b=d.fr.str.split().str.len()
    return d[(a<=MAXLEN)&(b<=MAXLEN)]
tr,va,te=ld(P_TR),ld(P_VA),ld(P_TE)
tr=cap(tr.drop_duplicates("en")).sample(frac=1,random_state=SEED).head(N)
va=cap(va)
print(len(tr),len(va),len(te))

In [ ]:
def mkv(S):
    c=Counter(w for s in S for w in s.split())
    i2w=["<pad>","<sos>","<eos>","<unk>"]+[w for w,n in c.items() if n>=MINF]
    return {w:i for i,w in enumerate(i2w)},i2w
s2i,i2s=mkv(tr.en)
t2i,i2t=mkv(tr.fr)
print("English vocab",len(i2s),"| French vocab",len(i2t))

In [ ]:
def pairs(d):
    return [([s2i.get(w,UNK) for w in a.split()],[t2i.get(w,UNK) for w in b.split()]) for a,b in zip(d.en,d.fr)]
def coll(bt):
    s,t=zip(*bt)
    sl=torch.tensor([len(i) for i in s])
    S=pad_sequence([torch.tensor(i) for i in s],batch_first=True)
    X=pad_sequence([torch.tensor([SOS]+i) for i in t],batch_first=True)
    Y=pad_sequence([torch.tensor(i+[EOS]) for i in t],batch_first=True)
    return S,sl,X,Y
trl=DataLoader(pairs(tr),batch_size=BS,shuffle=True,collate_fn=coll)
val=DataLoader(pairs(va),batch_size=BS,collate_fn=coll)
tel=DataLoader(pairs(te),batch_size=BS,collate_fn=coll)

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self,sv,tv,att):
        super().__init__()
        self.att=att
        self.se=nn.Embedding(sv,D,padding_idx=PAD)
        self.te=nn.Embedding(tv,D,padding_idx=PAD)
        self.enc=nn.LSTM(D,H,L,batch_first=True,dropout=DR)
        self.dec=nn.LSTM(D,H,L,batch_first=True,dropout=DR)
        self.out=nn.Linear(H,tv)
        self.dr=nn.Dropout(DR)
        if att:
            self.W=nn.Linear(H,H,bias=False)
            self.C=nn.Linear(2*H,H)
    def encode(self,s,sl):
        o,st=self.enc(pack_padded_sequence(self.dr(self.se(s)),sl.cpu(),batch_first=True,enforce_sorted=False))
        return pad_packed_sequence(o,batch_first=True,total_length=s.size(1))[0],st
    def decode(self,x,st,eo,mk):
        o,st=self.dec(self.dr(self.te(x)),st)
        if self.att:
            self.aw=torch.softmax(torch.bmm(self.W(o),eo.transpose(1,2)).masked_fill(mk[:,None,:],-1e9),-1)
            o=torch.tanh(self.C(torch.cat([torch.bmm(self.aw,eo),o],-1)))
        return self.out(self.dr(o)),st
    def forward(self,s,sl,x):
        eo,st=self.encode(s,sl)
        return self.decode(x,st,eo,s==PAD)[0]

In [ ]:
lf=nn.CrossEntropyLoss(ignore_index=PAD)
def ep(m,l,opt=None):
    m.train(opt is not None)
    s=n=0
    with torch.set_grad_enabled(opt is not None):
        for S,sl,X,Y in l:
            S,X,Y=S.to(DEV),X.to(DEV),Y.to(DEV)
            o=m(S,sl,X)
            loss=lf(o.reshape(-1,o.size(-1)),Y.reshape(-1))
            if opt is not None:
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(),1.0)
                opt.step()
            k=(Y!=PAD).sum().item()
            s+=loss.item()*k
            n+=k
    return s/n
def fit(att):
    torch.manual_seed(SEED)
    m=Seq2Seq(len(i2s),len(i2t),att).to(DEV)
    opt=torch.optim.Adam(m.parameters(),lr=LR)
    h={"train":[],"val":[]}
    bv,st,bad=1e9,None,0
    t0=time.time()
    for e in range(1,EPOCHS+1):
        h["train"].append(ep(m,trl,opt))
        h["val"].append(ep(m,val))
        print(f"{e:02d} train {h['train'][-1]:.4f} val {h['val'][-1]:.4f} val ppl {np.exp(h['val'][-1]):.2f}")
        if h["val"][-1]<bv:bv,st,bad=h["val"][-1],copy.deepcopy(m.state_dict()),0
        else:
            bad+=1
            if bad>=PAT:break
    sec=time.time()-t0
    m.load_state_dict(st)
    return m,h,sec
M={}
for k in ["No attention","Attention"]:
    print("\n"+k)
    M[k]=fit(k=="Attention")

In [ ]:
for i,k in enumerate(M):
    plt.plot(M[k][1]["train"],color=f"C{i}",label=k+" train")
    plt.plot(M[k][1]["val"],"--",color=f"C{i}",label=k+" validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy per token")
plt.legend()
plt.show()

In [ ]:
def beam(m,S,sl,k,a=1):
    B=len(S)
    eo,st=m.encode(S,sl)
    eo,mk=eo.repeat_interleave(k,0),(S==PAD).repeat_interleave(k,0)
    st=tuple(s.repeat_interleave(k,1) for s in st)
    x=torch.full((B*k,1),SOS,dtype=torch.long,device=DEV)
    sc=torch.zeros(B,k,device=DEV)
    sc[:,1:]=-1e9
    seq=torch.zeros(B*k,0,dtype=torch.long,device=DEV)
    fin=torch.zeros(B*k,dtype=torch.bool,device=DEV)
    ln=torch.zeros(B*k,device=DEV)
    base=(torch.arange(B,device=DEV)*k).unsqueeze(1)
    for _ in range(DMAX):
        y,st=m.decode(x,st,eo,mk)
        lp=torch.log_softmax(y[:,-1],-1)
        V=lp.size(-1)
        lp[fin]=-1e9
        lp[fin,PAD]=0
        sc,ix=(sc.reshape(-1,1)+lp).view(B,-1).topk(k,-1)
        sel=(base+ix//V).view(-1)
        x=(ix%V).view(-1,1)
        seq=torch.cat([seq[sel],x],1)
        st=tuple(s[:,sel] for s in st)
        pf=fin[sel]
        ln=ln[sel]+(~pf).float()
        fin=pf|(x[:,0]==EOS)
        if fin.all():break
    b=(sc/ln.view(B,k)**a).argmax(-1)
    return seq.view(B,k,-1)[torch.arange(B,device=DEV),b].tolist()
def samp(m,S,sl,k=10,T=1.0):
    eo,st=m.encode(S,sl)
    mk=S==PAD
    x=torch.full((len(S),1),SOS,dtype=torch.long,device=DEV)
    o=[]
    for _ in range(DMAX):
        y,st=m.decode(x,st,eo,mk)
        v,i=(y[:,-1]/T).topk(k,-1)
        x=i.gather(1,torch.multinomial(torch.softmax(v,-1),1))
        o.append(x)
        if (torch.cat(o,1)==EOS).any(1).all():break
    return torch.cat(o,1).tolist()
def run(m,f):
    m.eval()
    out=[]
    with torch.no_grad():
        for S,sl,_,_ in tel:
            for r in f(m,S.to(DEV),sl):
                r=r[:r.index(EOS)] if EOS in r else r
                out.append(" ".join(i2t[i] for i in r if i!=PAD))
    return out

In [ ]:
bl=lambda h,r:sacrebleu.corpus_bleu(list(h),[list(r)],tokenize="none",force=True).score
nw=lambda x:x.str.split().str.len().sum()
Rf=te.fr.tolist()
sl=te.en.str.split().str.len().to_numpy()
G={k:run(M[k][0],lambda m,S,sl:beam(m,S,sl,1)) for k in M}
print(pd.DataFrame({k:{"Test BLEU":bl(G[k],Rf),"Best val loss":min(M[k][1]["val"]),"Val ppl":np.exp(min(M[k][1]["val"])),"Params":sum(p.numel() for p in M[k][0].parameters()),"Epochs":len(M[k][1]["val"]),"Train sec":M[k][2]} for k in M}).T.round(3))
t=pd.DataFrame({"len":pd.cut(sl,[0,10,20,30,40,1e9],labels=["1-10","11-20","21-30","31-40","41+"]),"r":Rf,**G})
bt=pd.DataFrame({k:{b:bl(g[k],g.r) for b,g in t.groupby("len",observed=True)} for k in M})
bt["n"]=t.groupby("len",observed=True).size()
print(bt.round(2))
bt[list(M)].plot.bar()
plt.ylabel("BLEU")
plt.show()

In [ ]:
am=M["Attention"][0]
strat={
"Greedy":lambda m,S,sl:beam(m,S,sl,1),
"Beam 3":lambda m,S,sl:beam(m,S,sl,3),
"Beam 5":lambda m,S,sl:beam(m,S,sl,5),
"Beam 10":lambda m,S,sl:beam(m,S,sl,10),
"Beam 5 no length norm":lambda m,S,sl:beam(m,S,sl,5,0),
"Top-k sampling (k=10)":lambda m,S,sl:samp(m,S,sl,10,1.0)}
HY={}
rows=[]
for k,f in strat.items():
    torch.manual_seed(SEED)
    t0=time.time()
    HY[k]=run(am,f)
    rows.append({"Strategy":k,"BLEU":bl(HY[k],Rf),"len_ratio":nw(pd.Series(HY[k]))/nw(te.fr),"Sec":time.time()-t0})
print(pd.DataFrame(rows).set_index("Strategy").round(3))

In [ ]:
HY["No attention (greedy)"]=G["No attention"]
for i in np.random.RandomState(SEED).choice(len(te),5,replace=False):
    print("SRC:",te.en.iloc[i],"\nREF:",Rf[i])
    for k in HY:print(f"{k}:",HY[k][i])
    print()

In [ ]:
i=int(np.where((sl>=8)&(sl<=15))[0][0])
w=te.en.iloc[i].split()
S=torch.tensor([[s2i.get(x,UNK) for x in w]],device=DEV)
am.eval()
o,A=[],[]
with torch.no_grad():
    eo,st=am.encode(S,torch.tensor([len(w)]))
    x=torch.full((1,1),SOS,dtype=torch.long,device=DEV)
    for _ in range(DMAX):
        y,st=am.decode(x,st,eo,S==PAD)
        x=y[:,-1].argmax(-1,keepdim=True)
        if x.item()==EOS:break
        o.append(i2t[x.item()])
        A.append(am.aw[0,0].cpu().numpy())
plt.imshow(np.array(A),cmap="Blues")
plt.xticks(range(len(w)),w,rotation=90)
plt.yticks(range(len(o)),o)
plt.show()